# Stage 4 — temporal forensic analysis

This notebook calculates time-based evidence while explicitly respecting that the selected scenario covers one calendar day. Multi-day persistence is not inferred.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/temporal/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])
fraud=set(pd.read_csv(PROCESSED/'fraudulent_users.csv',dtype={'src_number':'string'})['src_number'].dropna().tolist())
source=cdr[['record_id','timestamp','src_number']].rename(columns={'src_number':'sim'})
destination=cdr[['record_id','timestamp','dst_number']].rename(columns={'dst_number':'sim'})
activity=pd.concat([source,destination],ignore_index=True).drop_duplicates(['sim','record_id']).sort_values(['sim','timestamp'])
activity['known_fraud']=activity.sim.isin(fraud); activity['date']=activity.timestamp.dt.date; activity['hour']=activity.timestamp.dt.hour; activity['is_night']=activity.hour.between(0,5)
print('SIM-event rows:',len(activity),'dates:',activity.date.nunique(),activity.date.unique())

In [ ]:
def temporal_row(group):
    timestamps=group.timestamp.sort_values()
    gaps=timestamps.diff().dt.total_seconds().dropna()
    positive=gaps[gaps>0]
    mean_gap=positive.mean() if len(positive) else np.nan
    std_gap=positive.std(ddof=0) if len(positive) else np.nan
    burstiness=((std_gap-mean_gap)/(std_gap+mean_gap)) if pd.notna(mean_gap) and pd.notna(std_gap) and (std_gap+mean_gap)>0 else np.nan
    hour_counts=group.hour.value_counts()
    return pd.Series({
        'active_days':group.date.nunique(), 'active_hours':group.hour.nunique(),
        'first_activity_timestamp':timestamps.min(), 'last_activity_timestamp':timestamps.max(),
        'activity_span_seconds':(timestamps.max()-timestamps.min()).total_seconds(),
        'activities_per_active_hour':len(group)/group.hour.nunique(),
        'activities_per_active_day':len(group)/group.date.nunique(),
        'hourly_activity_concentration':hour_counts.max()/len(group),
        'peak_activity_hour':hour_counts.idxmax(), 'max_hourly_activity':hour_counts.max(),
        'longest_inactivity_gap':positive.max() if len(positive) else np.nan,
        'shortest_inactivity_gap':positive.min() if len(positive) else np.nan,
        'night_activity_count':int(group.is_night.sum()), 'night_activity_ratio':group.is_night.mean(),
        'burstiness':burstiness, 'max_hourly_activity_ratio':hour_counts.max()/len(group),
        'hours_with_multiple_activities':int((hour_counts>1).sum()), 'known_fraud':bool(group.known_fraud.iloc[0])})

features=activity.groupby('sim',group_keys=False).apply(temporal_row).reset_index().rename(columns={'sim':'sim_number'})
features.to_csv(OUTPUT/'temporal_features.csv',index=False)
features.head()

In [ ]:
numeric=['active_days','active_hours','activity_span_seconds','activities_per_active_hour','activities_per_active_day','hourly_activity_concentration','max_hourly_activity','longest_inactivity_gap','shortest_inactivity_gap','night_activity_count','night_activity_ratio','burstiness','max_hourly_activity_ratio','hours_with_multiple_activities']
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')
for name,frame in [('all_sims',features),('known_fraudulent',features[features.known_fraud]),('known_legitimate',features[~features.known_fraud])]:
    table=frame[numeric].describe(percentiles=[.25,.5,.75]).T.rename(columns={'25%':'Q1','50%':'median','75%':'Q3'}); table['IQR']=table.Q3-table.Q1; table.to_csv(OUTPUT/f'temporal_descriptive_statistics_{name}.csv')
comparison=features.groupby('known_fraud',observed=False)[numeric].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']; comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'temporal_group_median_comparison.csv')
comparison

In [ ]:
hourly=activity.groupby(['hour','known_fraud']).size().unstack(fill_value=0).reindex(range(24),fill_value=0)
hourly.columns=['known_legitimate_events' if c is False else 'known_fraudulent_events' for c in hourly.columns]
hourly.to_csv(OUTPUT/'temporal_hourly_activity_distribution.csv')
print('distinct dates:',activity.date.nunique()); print('hourly distribution:'); print(hourly)

Interpretation rule: active days and day-to-day persistence are not discriminative here because the source covers one calendar day. Hourly concentration, activity span, gaps, burstiness, and night activity remain usable as short-window evidence, with undefined gap/burstiness values retained as NaN.